In [ ]:
from pathlib import Path
import sys
import warnings
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.max_columns", 70)
pd.set_option("display.width", 200)

# Google Drive mount happens only in Colab.
IN_COLAB = "google.colab" in sys.modules or Path("/content").exists()
if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        DATA_DIR = Path("/content/drive/MyDrive/Linguistics_feature")
    except (ImportError, RuntimeError):
        DATA_DIR = Path.cwd()
else:
    DATA_DIR = Path.cwd()

# EDIT ONLY IF YOUR CSV IS IN A DIFFERENT FOLDER.
AI_FEATURE_PATH = DATA_DIR / "raid_chatgpt_linguistic_features_v2.csv"
OUTPUT_DIR = DATA_DIR / "source_level_sensitivity_results"
FEATURE_COLS = [
    "word_count", "lexical_diversity", "avg_sentence_length",
    "sentence_length_std", "burstiness", "punctuation_density",
    "readability_flesch", "perplexity",
]
ALPHA = 0.05

print("Expected AI feature file:", AI_FEATURE_PATH)
print("Exists:", AI_FEATURE_PATH.exists())
if not AI_FEATURE_PATH.exists():
    raise FileNotFoundError(
        "The row-level AI feature CSV is not available. Upload/copy "
        "raid_chatgpt_linguistic_features_v2.csv into DATA_DIR, "
        "or edit AI_FEATURE_PATH above. This file was produced by your "
        "feature-extraction / final-analysis notebooks."
    )

Mounted at /content/drive
Expected AI feature file: /content/drive/MyDrive/Linguistics_feature/raid_chatgpt_linguistic_features_v2.csv
Exists: True


In [ ]:
required = ["id", "adv_source_id", "source_id", "attack", "domain"] + FEATURE_COLS
header = pd.read_csv(AI_FEATURE_PATH, nrows=0).columns.tolist()
missing = sorted(set(required) - set(header))
if missing:
    raise KeyError(f"Input feature CSV missing columns: {missing}")

ai = pd.read_csv(AI_FEATURE_PATH, usecols=required, low_memory=False)
for col in ["id", "adv_source_id", "source_id", "attack", "domain"]:
    ai[col] = ai[col].astype("string").str.strip()
for col in FEATURE_COLS:
    ai[col] = pd.to_numeric(ai[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

is_original = ai["attack"].fillna("").str.lower().eq("none")
originals = ai.loc[is_original].copy()
attacked = ai.loc[~is_original].copy()

if originals["id"].isna().any() or originals["id"].duplicated().any():
    raise ValueError("Original ChatGPT id must be nonmissing and unique. Resolve duplicates first.")
if originals["source_id"].isna().any():
    raise ValueError("Original rows with missing source_id cannot support source aggregation.")
if attacked[["adv_source_id", "source_id", "attack", "domain"]].isna().any().any():
    raise ValueError("Attacked rows have missing linking/group identifiers; inspect before proceeding.")

source_counts = (originals.groupby("source_id", observed=True)
                 .size().rename("N_original_texts").reset_index())
print("Original ChatGPT rows:", f"{len(originals):,}")
print("Unique source IDs:", f"{source_counts.shape[0]:,}")
print("Attacked rows:", f"{len(attacked):,}")
print("Sources with >1 original:", f"{(source_counts.N_original_texts > 1).sum():,}")
print("Maximum originals per source:", int(source_counts.N_original_texts.max()))
print("Originals per source — distribution:")
display(source_counts["N_original_texts"].value_counts().sort_index().rename_axis("Originals_per_source").reset_index(name="Sources"))

source_domains = originals.groupby("source_id", observed=True)["domain"].nunique()
print("Sources spanning multiple domains:", int((source_domains > 1).sum()))
if (source_domains > 1).any():
    print("CAUTION: the ALL scope pools source-level differences across domains for the same source ID.")

Original ChatGPT rows: 26,742
Unique source IDs: 13,371
Attacked rows: 294,162
Sources with >1 original: 13,371
Maximum originals per source: 2
Originals per source — distribution:


,Originals_per_source,Sources
0,2,13371


Sources spanning multiple domains: 0


In [ ]:
orig = originals[["id", "source_id", "domain"] + FEATURE_COLS].rename(columns={
    "id": "original_id", "source_id": "source_id_original",
    "domain": "domain_original",
    **{f: f"{f}_original" for f in FEATURE_COLS}
})
att = attacked[["id", "adv_source_id", "source_id", "attack", "domain"] + FEATURE_COLS].rename(columns={
    "id": "attacked_id", "source_id": "source_id_attacked",
    "domain": "domain_attacked",
    **{f: f"{f}_attacked" for f in FEATURE_COLS}
})

pairs = att.merge(orig, left_on="adv_source_id", right_on="original_id",
                  how="inner", validate="many_to_one")
coverage = len(pairs) / len(attacked) if len(attacked) else 0
if coverage < 0.99999:
    raise ValueError(f"Only {coverage:.2%} of attacked rows paired; check IDs before analysing.")
source_match = pairs["source_id_attacked"].eq(pairs["source_id_original"])
domain_match = pairs["domain_attacked"].eq(pairs["domain_original"])
if not source_match.all() or not domain_match.all():
    raise ValueError("Attacked and original source_id/domain mismatch. Stop and investigate.")
pairs["source_id"] = pairs["source_id_original"]
pairs["domain"] = pairs["domain_original"]

variants = pairs.groupby(["attack", "domain", "original_id"], observed=True).size()
print("Paired attacked rows:", f"{len(pairs):,}", "| coverage:", f"{coverage:.2%}")
print("Original/attack/domain entries with >1 attack variant:", int((variants > 1).sum()))
print("Attacks:", int(pairs.attack.nunique()), "| Domains:", int(pairs.domain.nunique()))
print("Unique sources per attack/domain (first 12 groups):")
display(pairs.groupby(["attack", "domain"], observed=True)["source_id"]
        .nunique().rename("N_unique_sources").reset_index().head(12))

Paired attacked rows: 294,162 | coverage: 100.00%
Original/attack/domain entries with >1 attack variant: 0
Attacks: 11 | Domains: 8
Unique sources per attack/domain (first 12 groups):


,attack,domain,N_unique_sources
0,alternative_spelling,abstracts,1766
1,alternative_spelling,books,1781
2,alternative_spelling,news,1780
3,alternative_spelling,poetry,1771
4,alternative_spelling,recipes,1772
5,alternative_spelling,reddit,1779
6,alternative_spelling,reviews,943
7,alternative_spelling,wiki,1779
8,article_deletion,abstracts,1766
9,article_deletion,books,1781


In [ ]:
def bh_adjust(p_values):
    p = np.asarray(p_values, dtype=float)
    out = np.full(len(p), np.nan)
    valid = np.isfinite(p)
    if not valid.any():
        return out
    pv = p[valid]
    order = np.argsort(pv)
    sorted_p = pv[order]
    m = len(sorted_p)
    adj = sorted_p * m / np.arange(1, m + 1)
    adj = np.clip(np.minimum.accumulate(adj[::-1])[::-1], 0, 1)
    restored = np.empty(m)
    restored[order] = adj
    out[np.where(valid)[0]] = restored
    return out


def paired_difference_stats(delta):
    """A single vector of attacked-minus-original paired differences."""
    d = np.asarray(delta, dtype=float)
    d = d[np.isfinite(d)]
    n = len(d)
    template = dict(n=n, mean=np.nan, dz=np.nan, ci_low=np.nan, ci_high=np.nan,
                    t_p=np.nan, wilcoxon_p=np.nan)
    if n == 0:
        return template
    template["mean"] = float(np.mean(d))
    if n < 2:
        return template
    sd = np.std(d, ddof=1)
    if not np.isfinite(sd) or np.isclose(sd, 0):
        template["dz"] = 0.0 if np.isclose(np.mean(d), 0) else np.nan
    else:
        template["dz"] = float(np.mean(d) / sd)
    se = stats.sem(d)
    if np.isfinite(se):
        critical = stats.t.ppf(0.975, n - 1)
        template["ci_low"] = float(np.mean(d) - critical * se)
        template["ci_high"] = float(np.mean(d) + critical * se)
    # In the original notebook ttest_rel(before, after) is two-sided;
    # p-values are equal to one-sample t-test on (after-before).
    try:
        template["t_p"] = float(stats.ttest_1samp(d, 0).pvalue)
    except (ValueError, FloatingPointError):
        pass
    try:
        template["wilcoxon_p"] = (1.0 if np.allclose(d, 0)
                                  else float(stats.wilcoxon(d, zero_method="wilcox").pvalue))
    except (ValueError, FloatingPointError):
        pass
    return template


def compare_feature(subset, feature, attack, scope):
    """First average variants per original, then differences per source."""
    c_orig, c_att = f"{feature}_original", f"{feature}_attacked"
    temp = subset[["original_id", "source_id", c_orig, c_att]].copy()
    for c in (c_orig, c_att):
        temp[c] = pd.to_numeric(temp[c], errors="coerce").replace([np.inf, -np.inf], np.nan)

    # Match the original pipeline's per-original averaging of attacked variants.
    by_original = (temp.groupby(["original_id", "source_id"], as_index=False,
                                dropna=False, observed=True)
                   .agg(before=(c_orig, "first"), after=(c_att, "mean")))
    by_original = by_original.dropna(subset=["before", "after", "source_id"])
    by_original["delta"] = by_original["after"] - by_original["before"]

    # Primary result: one difference per original document pair.
    doc = paired_difference_stats(by_original["delta"].to_numpy())

    # Sensitivity result: ONE equal-weighted difference per source.
    by_source = (by_original.groupby("source_id", as_index=False, observed=True)
                 .agg(delta=("delta", "mean"),
                      n_original_pairs=("original_id", "size")))
    src = paired_difference_stats(by_source["delta"].to_numpy())

    record = dict(Attack=attack, Domain=scope, Linguistic_Feature=feature,
                  N_Document_Pairs=doc["n"], N_Unique_Sources=src["n"],
                  N_Sources_With_Multiple_Originals=int((by_source.n_original_pairs > 1).sum()),
                  Max_Originals_Per_Source=int(by_source.n_original_pairs.max()) if len(by_source) else 0)
    for prefix, values in [("Document", doc), ("Source", src)]:
        record.update({
            f"{prefix}_Mean_Delta": values["mean"],
            f"{prefix}_Cohens_dz": values["dz"],
            f"{prefix}_Abs_dz": abs(values["dz"]) if np.isfinite(values["dz"]) else np.nan,
            f"{prefix}_Delta_95CI_Low": values["ci_low"],
            f"{prefix}_Delta_95CI_High": values["ci_high"],
            f"{prefix}_Paired_t_p": values["t_p"],
            f"{prefix}_Wilcoxon_p": values["wilcoxon_p"],
        })
    return record

In [ ]:
results = []
attacks = sorted(pairs["attack"].dropna().unique().tolist())
for attack in attacks:
    attack_pairs = pairs.loc[pairs["attack"].eq(attack)]
    scopes = ["ALL"] + sorted(attack_pairs["domain"].dropna().unique().tolist())
    for scope in scopes:
        group = (attack_pairs if scope == "ALL"
                 else attack_pairs.loc[attack_pairs["domain"].eq(scope)])
        for feature in FEATURE_COLS:
            results.append(compare_feature(group, feature, attack, scope))

comparison = pd.DataFrame(results)
if comparison.empty:
    raise RuntimeError("No comparisons were computed.")

# FDR families: one attack and one analysis scope, eight features, separately by test.
for prefix in ["Document", "Source"]:
    for test in ["Paired_t", "Wilcoxon"]:
        raw = f"{prefix}_{test}_p"
        output = f"{prefix}_{test}_FDR"
        comparison[output] = comparison.groupby(["Attack", "Domain"], sort=False)[raw].transform(
            lambda x: bh_adjust(x.to_numpy())
        )
    comparison[f"{prefix}_Wilcoxon_Significant"] = comparison[f"{prefix}_Wilcoxon_FDR"].lt(ALPHA)
    comparison[f"{prefix}_t_Significant"] = comparison[f"{prefix}_Paired_t_FDR"].lt(ALPHA)

comparison["Wilcoxon_Significance_Changed"] = (
    comparison["Document_Wilcoxon_Significant"] != comparison["Source_Wilcoxon_Significant"]
)
comparison["Effect_Direction_Changed"] = (
    (np.sign(comparison["Document_Cohens_dz"]) != np.sign(comparison["Source_Cohens_dz"]))
    & comparison[["Document_Cohens_dz", "Source_Cohens_dz"]].notna().all(axis=1)
)
comparison["Abs_dz_Change_Source_minus_Document"] = comparison["Source_Abs_dz"] - comparison["Document_Abs_dz"]
comparison["N_Reduction_Pct"] = np.where(
    comparison["N_Document_Pairs"].gt(0),
    100 * (1 - comparison["N_Unique_Sources"] / comparison["N_Document_Pairs"]),
    np.nan,
)

overall_compare = comparison.loc[comparison.Domain.eq("ALL")].copy()
domain_compare = comparison.loc[~comparison.Domain.eq("ALL")].copy()
print(f"Overall attack–feature rows: {len(overall_compare):,}")
print(f"Attack–domain–feature rows: {len(domain_compare):,}")
print("Expected with 11 attacks × 8 features, and 11 × 8 × 8:", 88, 704)
if len(overall_compare) != 88 or len(domain_compare) != 704:
    print("NOTE: data does not contain all expected 11 attacks / 8 domains. Check coverage.")
display(domain_compare[["Attack", "Domain", "Linguistic_Feature", "N_Document_Pairs",
                        "N_Unique_Sources", "Document_Cohens_dz", "Source_Cohens_dz",
                        "Document_Wilcoxon_FDR", "Source_Wilcoxon_FDR",
                        "Wilcoxon_Significance_Changed"]].head(12).round(5))

Overall attack–feature rows: 88
Attack–domain–feature rows: 704
Expected with 11 attacks × 8 features, and 11 × 8 × 8: 88 704


,Attack,Domain,Linguistic_Feature,N_Document_Pairs,N_Unique_Sources,Document_Cohens_dz,Source_Cohens_dz,Document_Wilcoxon_FDR,Source_Wilcoxon_FDR,Wilcoxon_Significance_Changed
8,alternative_spelling,abstracts,word_count,3532,1766,0.00000,0.00000,1.00000,1.00000,False
9,alternative_spelling,abstracts,lexical_diversity,3532,1766,0.08867,0.10785,0.00001,0.00004,False
10,alternative_spelling,abstracts,avg_sentence_length,3532,1766,0.00000,0.00000,1.00000,1.00000,False
11,alternative_spelling,abstracts,sentence_length_std,3532,1766,0.00000,0.00000,1.00000,1.00000,False
12,alternative_spelling,abstracts,burstiness,3532,1766,0.00000,0.00000,1.00000,1.00000,False
13,alternative_spelling,abstracts,punctuation_density,3532,1766,-0.53344,-0.57773,0.00000,0.00000,False
14,alternative_spelling,abstracts,readability_flesch,3532,1766,0.15073,0.15891,0.00000,0.00000,False
15,alternative_spelling,abstracts,perplexity,3532,1766,0.55447,0.65853,0.00000,0.00000,False
16,alternative_spelling,books,word_count,3562,1781,0.00000,0.00000,1.00000,1.00000,False
17,alternative_spelling,books,lexical_diversity,3562,1781,0.05852,0.07091,0.00307,0.00575,False


In [ ]:
def summarize_scope(df, scope_name):
    valid = df[["Document_Wilcoxon_FDR", "Source_Wilcoxon_FDR"]].notna().all(axis=1)
    n_valid = int(valid.sum())
    return dict(
        Scope=scope_name,
        N_Feature_Comparisons=len(df),
        N_Valid_FDR_Both=n_valid,
        N_Document_FDR_Significant=int((df.loc[valid, "Document_Wilcoxon_FDR"] < ALPHA).sum()),
        N_Source_FDR_Significant=int((df.loc[valid, "Source_Wilcoxon_FDR"] < ALPHA).sum()),
        N_Significance_Decisions_Changed=int(df.loc[valid, "Wilcoxon_Significance_Changed"].sum()),
        N_Effect_Directions_Changed=int(df.Effect_Direction_Changed.sum()),
        Median_Document_Abs_dz=float(df.Document_Abs_dz.median()),
        Median_Source_Abs_dz=float(df.Source_Abs_dz.median()),
        Median_Source_Unit_Reduction_pct=float(df.N_Reduction_Pct.median()),
    )

scope_summary = pd.DataFrame([
    summarize_scope(overall_compare, "Overall attack-level"),
    summarize_scope(domain_compare, "Attack-domain-level"),
])
display(scope_summary.round(4))

feature_summary = (domain_compare.groupby("Linguistic_Feature", as_index=False)
    .agg(
        N_Attack_Domain=("Attack", "size"),
        Document_FDR_Significant=("Document_Wilcoxon_Significant", "sum"),
        Source_FDR_Significant=("Source_Wilcoxon_Significant", "sum"),
        Significance_Changes=("Wilcoxon_Significance_Changed", "sum"),
        Median_Document_Abs_dz=("Document_Abs_dz", "median"),
        Median_Source_Abs_dz=("Source_Abs_dz", "median"),
        Mean_Document_Abs_dz=("Document_Abs_dz", "mean"),
        Mean_Source_Abs_dz=("Source_Abs_dz", "mean"),
    )
)
feature_summary["Document_Significant_pct"] = 100 * feature_summary.Document_FDR_Significant / feature_summary.N_Attack_Domain
feature_summary["Source_Significant_pct"] = 100 * feature_summary.Source_FDR_Significant / feature_summary.N_Attack_Domain
feature_summary["Median_Abs_dz_Difference"] = feature_summary.Median_Source_Abs_dz - feature_summary.Median_Document_Abs_dz
feature_summary = feature_summary.sort_values("Median_Document_Abs_dz", ascending=False)
display(feature_summary.round(4))

changed_cases = comparison.loc[
    comparison.Wilcoxon_Significance_Changed | comparison.Effect_Direction_Changed
].sort_values(["Attack", "Domain", "Linguistic_Feature"])
print("Comparisons with changed Wilcoxon significance OR changed d_z direction:", len(changed_cases))
display(changed_cases[["Attack", "Domain", "Linguistic_Feature",
                       "N_Document_Pairs", "N_Unique_Sources",
                       "Document_Cohens_dz", "Source_Cohens_dz",
                       "Document_Wilcoxon_FDR", "Source_Wilcoxon_FDR",
                       "Wilcoxon_Significance_Changed", "Effect_Direction_Changed"]]
        .head(30).round(5))

,Scope,N_Feature_Comparisons,N_Valid_FDR_Both,N_Document_FDR_Significant,N_Source_FDR_Significant,N_Significance_Decisions_Changed,N_Effect_Directions_Changed,Median_Document_Abs_dz,Median_Source_Abs_dz,Median_Source_Unit_Reduction_pct
0,Overall attack-level,88,88,48,48,0,0,0.0856,0.0980,50.0
1,Attack-domain-level,704,704,371,366,5,0,0.0571,0.0709,50.0


,Linguistic_Feature,N_Attack_Domain,Document_FDR_Significant,Source_FDR_Significant,Significance_Changes,Median_Document_Abs_dz,Median_Source_Abs_dz,Mean_Document_Abs_dz,Mean_Source_Abs_dz,Document_Significant_pct,Source_Significant_pct,Median_Abs_dz_Difference
3,perplexity,88,88,88,0,1.7514,2.3250,2.0090,2.5307,100.0000,100.0000,0.5736
4,punctuation_density,88,62,62,0,0.5009,0.5794,1.6731,1.9962,70.4545,70.4545,0.0785
2,lexical_diversity,88,61,60,1,0.2379,0.2718,1.8033,2.2149,69.3182,68.1818,0.0339
5,readability_flesch,88,56,55,1,0.1591,0.1935,1.4797,1.7192,63.6364,62.5000,0.0344
0,avg_sentence_length,88,28,27,1,0.0000,0.0000,0.7776,0.9348,31.8182,30.6818,0.0000
1,burstiness,88,25,24,1,0.0000,0.0000,0.3979,0.4936,28.4091,27.2727,0.0000
6,sentence_length_std,88,27,26,1,0.0000,0.0000,0.4113,0.5276,30.6818,29.5455,0.0000
7,word_count,88,24,24,0,0.0000,0.0000,0.8250,1.0021,27.2727,27.2727,0.0000


Comparisons with changed Wilcoxon significance OR changed d_z direction: 5


,Attack,Domain,Linguistic_Feature,N_Document_Pairs,N_Unique_Sources,Document_Cohens_dz,Source_Cohens_dz,Document_Wilcoxon_FDR,Source_Wilcoxon_FDR,Wilcoxon_Significance_Changed,Effect_Direction_Changed
25,alternative_spelling,news,lexical_diversity,3560,1780,0.04521,0.05842,0.01404,0.06197,True,False
242,insert_paragraphs,news,avg_sentence_length,3560,1780,-0.03697,-0.04748,0.03694,0.05749,True,False
244,insert_paragraphs,news,burstiness,3560,1780,0.03860,0.04485,0.03694,0.05749,True,False
246,insert_paragraphs,news,readability_flesch,3560,1780,0.03697,0.04748,0.03694,0.05749,True,False
243,insert_paragraphs,news,sentence_length_std,3560,1780,0.03600,0.04095,0.03694,0.05749,True,False


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
outputs = {
    "01_originals_per_source.csv": source_counts,
    "02_overall_document_vs_source.csv": overall_compare,
    "03_domain_document_vs_source.csv": domain_compare,
    "04_source_sensitivity_feature_summary.csv": feature_summary,
    "05_source_sensitivity_scope_summary.csv": scope_summary,
    "06_significance_or_direction_changes.csv": changed_cases,
}
for name, df in outputs.items():
    out_path = OUTPUT_DIR / name
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path} | {len(df):,} rows")

# A short report that copies numbers from computed output (not hypothetical values).
report = [
    "# RAID source-level sensitivity analysis\n",
    f"Input: {AI_FEATURE_PATH.name}",
    f"Original rows: {len(originals):,}; distinct sources: {originals.source_id.nunique():,}",
    f"Paired attacked rows: {len(pairs):,}; pairing coverage: {coverage:.2%}",
    "Paired differences: attacked minus original; source differences are the mean of valid original-level differences per source.",
    "Wilcoxon and paired t-test p-values were adjusted with BH separately across eight features per attack/scope.\n",
]
for row in scope_summary.to_dict("records"):
    report.extend([
        f"## {row['Scope']}",
        f"Compared features: {row['N_Feature_Comparisons']}",
        f"FDR-valid in both analyses: {row['N_Valid_FDR_Both']}",
        f"FDR-significant comparisons: document = {row['N_Document_FDR_Significant']}, source = {row['N_Source_FDR_Significant']}",
        f"Changed Wilcoxon significance decisions = {row['N_Significance_Decisions_Changed']}",
        f"Changed d_z directions = {row['N_Effect_Directions_Changed']}",
        f"Median |d_z|: document = {row['Median_Document_Abs_dz']:.4f}, source = {row['Median_Source_Abs_dz']:.4f}",
        "",
    ])
report.extend([
    "## Interpretation guidance",
    "These statistics examine sensitivity to repeated sources; they do not establish statistical equivalence or detector accuracy.",
    "Report substantive changes in FDR decisions, direction, and effect magnitude; if differences are large, revisit the main statistical interpretation.",
])
report_path = OUTPUT_DIR / "07_source_sensitivity_summary.md"
report_path.write_text("\n".join(report), encoding="utf8")
print("Saved:", report_path)

Saved: /content/drive/MyDrive/Linguistics_feature/source_level_sensitivity_results/01_originals_per_source.csv | 13,371 rows
Saved: /content/drive/MyDrive/Linguistics_feature/source_level_sensitivity_results/02_overall_document_vs_source.csv | 88 rows
Saved: /content/drive/MyDrive/Linguistics_feature/source_level_sensitivity_results/03_domain_document_vs_source.csv | 704 rows
Saved: /content/drive/MyDrive/Linguistics_feature/source_level_sensitivity_results/04_source_sensitivity_feature_summary.csv | 8 rows
Saved: /content/drive/MyDrive/Linguistics_feature/source_level_sensitivity_results/05_source_sensitivity_scope_summary.csv | 2 rows
Saved: /content/drive/MyDrive/Linguistics_feature/source_level_sensitivity_results/06_significance_or_direction_changes.csv | 5 rows
Saved: /content/drive/MyDrive/Linguistics_feature/source_level_sensitivity_results/07_source_sensitivity_summary.md
